# Limpieza y transformación del conjunto de datos

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tabulate import tabulate
from google.colab import files

df= pd.read_csv("Encuesta2023.csv", encoding='latin1')
dicpreg= pd.read_csv("dicpreguntas.csv", encoding='latin1')
dicvars= pd.read_csv("dicvariables.csv", encoding='latin1')

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21032 entries, 0 to 21031
Columns: 432 entries, CCAA to CUADROS_DEPRESIVOS
dtypes: float64(284), int64(148)
memory usage: 69.3 MB


In [ ]:
print("Dimensiones del conjunto de datos:", df.shape)

Dimensiones del conjunto de datos: (21032, 432)


In [ ]:
dicpreg.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 24 entries, 0 to 23
Data columns (total 4 columns):
 #   Column                      Non-Null Count  Dtype 
---  ------                      --------------  ----- 
 0   Ítem                        24 non-null     int64 
 1   Variable                    24 non-null     object
 2   Diccionario de la variable  24 non-null     object
 3   Descripción                 24 non-null     object
dtypes: int64(1), object(3)
memory usage: 900.0+ bytes


#Preparación de los datos:

# Eliminar variables no seleccionadas

In [ ]:
varsel= dicpreg['Variable'].unique().tolist()

In [ ]:
varsel[:5]

['CCAA', 'SEXOa', 'EDADa', 'A1a', 'CLASE_PR']

In [ ]:
df_selec= df.loc[:,df.columns.isin(varsel)]

In [ ]:
print("Dimension conjunto filtrado:", df_selec.shape)

Dimension conjunto filtrado: (21032, 24)


In [ ]:
df_selec.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21032 entries, 0 to 21031
Data columns (total 24 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   CCAA      21032 non-null  int64  
 1   SEXOa     21032 non-null  int64  
 2   EDADa     21032 non-null  int64  
 3   A1a       21032 non-null  int64  
 4   NIVEST    21032 non-null  int64  
 5   C1        21032 non-null  int64  
 6   C2        21032 non-null  int64  
 7   C5a_1     21032 non-null  int64  
 8   C5a_12    21032 non-null  int64  
 9   C5a_15    21032 non-null  int64  
 10  N1        21032 non-null  int64  
 11  N2        21032 non-null  int64  
 12  O2        21032 non-null  int64  
 13  O10_1     21032 non-null  int64  
 14  P1_1      21032 non-null  int64  
 15  P1_2      21032 non-null  int64  
 16  P1_4      21032 non-null  int64  
 17  P1_7      21032 non-null  int64  
 18  P1_9      21032 non-null  int64  
 19  P1_10     21032 non-null  int64  
 20  P1_12     21032 non-null  in

# Limpieza

In [ ]:
df_selec1 = df_selec.copy()

In [ ]:
# Ver los diccionarios de variables y de preguntas

dicpreg.info()
dicvars.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 24 entries, 0 to 23
Data columns (total 4 columns):
 #   Column                      Non-Null Count  Dtype 
---  ------                      --------------  ----- 
 0   Ítem                        24 non-null     int64 
 1   Variable                    24 non-null     object
 2   Diccionario de la variable  24 non-null     object
 3   Descripción                 24 non-null     object
dtypes: int64(1), object(3)
memory usage: 900.0+ bytes
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 847 entries, 0 to 846
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   Variable     847 non-null    object
 1   Código       847 non-null    int64 
 2   Descripción  847 non-null    object
dtypes: int64(1), object(2)
memory usage: 20.0+ KB


In [ ]:
# Para el mapeo de las columnas del conjunto de datos en los diccionarios, se limpian los espacios en blanco

dicvars.columns= dicvars.columns.str.strip()
dicpreg.columns= dicpreg.columns.str.strip()

for df in [dicpreg, dicvars]:
  for col in df.columns:
    df[col]= df[col].astype(str).str.strip()

In [ ]:
# Renombrado de variables para interpretabilidad

dic_mapa= dict(zip(dicpreg['Variable'], dicpreg['Descripción']))
df_selec1.rename(columns=dic_mapa, inplace=True)



In [ ]:
df_selec1.columns

Index(['Comunidad Autónoma de residencia',
       'Identificación del adulto seleccionado: Sexo 01 a 15',
       'Identificación del adulto seleccionado: Edad. De 015 a 120',
       'País de nacimiento', 'Nivel de estudios del adulto seleccionado',
       'Estado de salud percibido en los últimos 12 meses',
       'Enfermedad o problema de salud crónicos o de larga duración',
       'Ha padecido alguna vez: Tensión alta',
       'Ha padecido alguna vez: Diabetes',
       'Ha padecido alguna vez: Colesterol alto', 'Altura en cm (50 a 220)',
       'Peso en kg (25 a 180)',
       'Frecuencia con la que realiza alguna actividad física en su tiempo libre',
       'Sedentarismo : tiempo que permanece sentado a lo largo de un día normal. Horas',
       'Frecuencia de consumo de fruta fresca (excluyendo zumos)',
       'Frecuencia de consumo de carne (pollo, ternera, cerdo, cordero, etc.)',
       'Frecuencia de consumo de pescado',
       'Frecuencia de consumo de verduras, ensaladas y horta

# Variables de frecuencia y cantidad consumo

In [ ]:
# Eliminacion de no contesta o no consta, que corresponden a 9 en las variables de frecuencia de alimentación para reemplazarlas con NAs.

var_frec_alm= ["Frecuencia de consumo de carne (pollo, ternera, cerdo, cordero, etc.)",
                 "Frecuencia de consumo de productos lácteos (leche, queso, yogur)",
                 "Frecuencia de consumo de pescado", "Frecuencia de consumo de fruta fresca (excluyendo zumos)",
                 "Frecuencia de consumo de embutidos y fiambres", "Frecuencia de consumo de verduras, ensaladas y hortalizas",
                 "Frecuencia de consumo de refrescos con azúcar"]


In [ ]:
for col in var_frec_alm:
    df_selec1[col.strip()]= df_selec1[col.strip()].replace(9, np.nan)

In [ ]:
# Las frecuencias estan de forma 1:Una o más veces al día 2:De 4 a 6 veces a la semana, 3:Tres veces a la semana etc
# Para mejorar interpretabilidad, se mapean esas correspondencias para que se ajuste a un numero de veces por semana p.e 10 veces a la semana

frec_alm= {1: 10, 2: 5, 3: 3, 4: 1.5, 5: 0.5, 6:0}

for col in var_frec_alm:
  df_selec1[col+"_frec"]= df_selec1[col].map(frec_alm)

In [ ]:
cols_frec= [col for col in df_selec1.columns if col.endswith("_frec")]
print(tabulate(df_selec1[cols_frec].describe().T, headers='keys', tablefmt='psql'))

+----------------------------------------------------------------------------+---------+---------+---------+-------+-------+-------+-------+-------+
|                                                                            |   count |    mean |     std |   min |   25% |   50% |   75% |   max |
|----------------------------------------------------------------------------+---------+---------+---------+-------+-------+-------+-------+-------|
| Frecuencia de consumo de carne (pollo, ternera, cerdo, cordero, etc.)_frec |   20917 | 3.83896 | 2.55344 |     0 |   1.5 |   3   |   5   |    10 |
| Frecuencia de consumo de productos lácteos (leche, queso, yogur)_frec      |   20917 | 7.8965  | 3.24428 |     0 |   5   |  10   |  10   |    10 |
| Frecuencia de consumo de pescado_frec                                      |   20909 | 2.15429 | 1.59057 |     0 |   1.5 |   1.5 |   3   |    10 |
| Frecuencia de consumo de fruta fresca (excluyendo zumos)_frec              |   20915 | 7.0076  | 3.5321 

Consumo de alcohol-En cantidad

In [ ]:
# Se copia la variable de consumo de alcohol

df_selec1['Alcohol_cantidad']=df_selec1['Variable derivada: Consumo medio diario de alcohol semanal (lunes a domingo)']
df_selec1['Alcohol_cantidad']= df_selec1['Alcohol_cantidad'].replace(999, np.nan)


# Variables de salud

In [ ]:
salud_var={1:"Muy bueno", 2: "Bueno", 3:"Regular",4:"Malo", 5:"Muy malo"}

df_selec1['Salud percibida_cat']= df_selec1['Estado de salud percibido en los últimos 12 meses'].map(salud_var)

In [ ]:
df_selec1['Salud percibida_cat'].isna().sum()

np.int64(0)

In [ ]:
print(tabulate(df_selec1['Salud percibida_cat'].describe().to_frame(), headers='keys', tablefmt='psql'))

+--------+-----------------------+
|        | Salud percibida_cat   |
|--------+-----------------------|
| count  | 21032                 |
| unique | 5                     |
| top    | Bueno                 |
| freq   | 10207                 |
+--------+-----------------------+


Variables binarias enfermedades

In [ ]:
# Eliminacion de no contesta o no consta, que corresponden a 9 en las variables binarias

var_bin_enf= ["Enfermedad o problema de salud crónicos o de larga duración","Ha padecido alguna vez: Tensión alta",
              "Ha padecido alguna vez: Diabetes", "Ha padecido alguna vez: Colesterol alto"]

In [ ]:
for col in var_bin_enf:
    df_selec1[col.strip()]= df_selec1[col.strip()].replace(9, np.nan)

In [ ]:
# Las frecuencias estan de forma 1:Si 2: No, 9: No contesta
# Para mejorar interpretabilidad, se mapean esas correspondencias para que se ajuste a binario

for col in var_bin_enf:
  df_selec1[col+"_bin"]= df_selec1[col].replace({1: 1, 2: 0})

In [ ]:
cols_bin= [col for col in df_selec1.columns if col.endswith("_bin")]

In [ ]:
for col in cols_bin:
    uniques= df_selec1[col].unique().tolist()
    print(uniques)

[1.0, 0.0, nan]
[1.0, 0.0, nan]
[0.0, 1.0, nan]
[0.0, 1.0, nan]


In [ ]:
print(tabulate(df_selec1[cols_bin].describe().T, headers='keys', tablefmt='psql'))

+-----------------------------------------------------------------+---------+-----------+----------+-------+-------+-------+-------+-------+
|                                                                 |   count |      mean |      std |   min |   25% |   50% |   75% |   max |
|-----------------------------------------------------------------+---------+-----------+----------+-------+-------+-------+-------+-------|
| Enfermedad o problema de salud crónicos o de larga duración_bin |   20780 | 0.636429  | 0.481039 |     0 |     0 |     1 |     1 |     1 |
| Ha padecido alguna vez: Tensión alta_bin                        |   20724 | 0.292077  | 0.454728 |     0 |     0 |     0 |     1 |     1 |
| Ha padecido alguna vez: Diabetes_bin                            |   20872 | 0.0966366 | 0.295469 |     0 |     0 |     0 |     0 |     1 |
| Ha padecido alguna vez: Colesterol alto_bin                     |   20788 | 0.261689  | 0.439565 |     0 |     0 |     0 |     1 |     1 |
+------------

In [ ]:
df_selec1.columns

Index(['Comunidad Autónoma de residencia',
       'Identificación del adulto seleccionado: Sexo 01 a 15',
       'Identificación del adulto seleccionado: Edad. De 015 a 120',
       'País de nacimiento', 'Nivel de estudios del adulto seleccionado',
       'Estado de salud percibido en los últimos 12 meses',
       'Enfermedad o problema de salud crónicos o de larga duración',
       'Ha padecido alguna vez: Tensión alta',
       'Ha padecido alguna vez: Diabetes',
       'Ha padecido alguna vez: Colesterol alto', 'Altura en cm (50 a 220)',
       'Peso en kg (25 a 180)',
       'Frecuencia con la que realiza alguna actividad física en su tiempo libre',
       'Sedentarismo : tiempo que permanece sentado a lo largo de un día normal. Horas',
       'Frecuencia de consumo de fruta fresca (excluyendo zumos)',
       'Frecuencia de consumo de carne (pollo, ternera, cerdo, cordero, etc.)',
       'Frecuencia de consumo de pescado',
       'Frecuencia de consumo de verduras, ensaladas y horta

Creacion de variables adicionales con las enfermedades -Comorbilidad

In [ ]:
# Para crear una variable de comorbilidad,se suman las columnas en las que hay enfermedades cronicas
# Siendo la comorbilidad mas alta 4 (Ha tenido las 4 enfermedades) y minimo 0, no ha tenido ninguna

df_selec1["Comorbilidad"]= df_selec1[["Enfermedad o problema de salud crónicos o de larga duración_bin","Ha padecido alguna vez: Tensión alta_bin",
              "Ha padecido alguna vez: Diabetes_bin", "Ha padecido alguna vez: Colesterol alto_bin"]].sum(axis=1)

In [ ]:
print(tabulate(df_selec1["Comorbilidad"].describe().to_frame(), headers='keys', tablefmt='psql'))

+-------+----------------+
|       |   Comorbilidad |
|-------+----------------|
| count |    21032       |
| mean  |        1.27116 |
| std   |        1.15019 |
| min   |        0       |
| 25%   |        0       |
| 50%   |        1       |
| 75%   |        2       |
| max   |        4       |
+-------+----------------+


# Determinantes de salud

In [ ]:
# Ajuste de variables de peso y altura. Se reemplaza 999 por NAs

df_selec1['Peso']= df_selec1['Peso en kg (25 a 180)'].replace(999, np.nan)
df_selec1['Altura']= df_selec1['Altura en cm (50 a 220)'].replace(999, np.nan)

In [ ]:
# IMC numerico

df_selec1["IMC"]= df_selec1["Peso"]/((df_selec1["Altura"]/100)**2)

**Variable de actividad fisica**

In [ ]:
# La variable de actividad fisica es clave para el estudio, esta de la forma 1.No hago ejercicio. El tiempo libre lo ocupo de forma casi completamente sedentaria
# 2. Hago alguna actividad física o deportiva ocasional 3. Hago actividad física varias veces al mes etc por lo que tambien se debe ajustar
# Tambien se hace el reemplazo por NAs de los " No constest o no Consta identificados con 9"

act_fisica= {1: "Sedentario", 2: "Ocasional", 3:"Regular", 4: "Activo" }

df_selec1['Actividad_física_cat']= df_selec1['Frecuencia con la que realiza alguna actividad física en su tiempo libre'].replace({9: np.nan}).map(act_fisica)


In [ ]:
print(tabulate(df_selec1['Actividad_física_cat'].describe().to_frame(), headers='keys', tablefmt='psql'))

+--------+------------------------+
|        | Actividad_física_cat   |
|--------+------------------------|
| count  | 20761                  |
| unique | 4                      |
| top    | Ocasional              |
| freq   | 8541                   |
+--------+------------------------+


**Horas sentado**

In [ ]:
print(tabulate(df_selec1['Sedentarismo : tiempo que permanece sentado a lo largo de un día normal. Horas'].describe().to_frame(), headers='keys', tablefmt='psql'))

+-------+----------------------------------------------------------------------------------+
|       |   Sedentarismo : tiempo que permanece sentado a lo largo de un día normal. Horas |
|-------+----------------------------------------------------------------------------------|
| count |                                                                       21032      |
| mean  |                                                                          13.3919 |
| std   |                                                                          26.9331 |
| min   |                                                                           0      |
| 25%   |                                                                           3      |
| 50%   |                                                                           5      |
| 75%   |                                                                           8      |
| max   |                                                             

In [ ]:
# Se encuentran valores de "no contesta" =99 por lo que se reemplzan por NaN
df_selec1['Sedentarismo_horas_dia']=df_selec1['Sedentarismo : tiempo que permanece sentado a lo largo de un día normal. Horas'].replace(99, np.nan)


In [ ]:
df_selec1['Sedentarismo_horas_dia'].isna().sum()

np.int64(1868)

IMC Categoria

In [ ]:
# Se mapea y copia la columna de IMC

dicpreg['Descripcion']= dicpreg['Descripción'].astype(str).str.strip()

col_imc= 'Variable derivada: Índice de masa corporal (IMC)'
fila_imc= dicpreg[dicpreg['Descripción']== col_imc]

nom_imc= fila_imc['Diccionario de la variable'].iloc[0]

map_imc= dicvars.loc[dicvars['Variable']==nom_imc,['Código','Descripción']]
map_imc['Código']= map_imc['Código'].astype(str)
mapeo_imc= map_imc.set_index('Código')['Descripción'].to_dict()
df_selec1['IMC_cat']= df_selec1[col_imc].astype(str).map(mapeo_imc)


In [ ]:
df_selec1['IMC_cat']=df_selec1['IMC_cat'].replace('No consta', np.nan)

# Variables sociodemográficas

In [ ]:
var_soc= ['Comunidad Autónoma de residencia',
       'Identificación del adulto seleccionado: Sexo 01 a 15',
       'País de nacimiento', 'Nivel de estudios del adulto seleccionado',
       'Variable derivada: Clase social basada en la ocupación de la persona de referencia']

In [ ]:
# Reemplazo de las variables sociodemograficas, excluyendo la edad

for col in var_soc:

  fila_dic= dicpreg.loc[dicpreg['Descripción']==col]
  nom_dic= fila_dic['Diccionario de la variable'].iloc[0]
  mapeo= dicvars.loc[dicvars['Variable']==nom_dic,['Código','Descripción']]
  mapeo['Código']= mapeo['Código'].astype(str)
  mapeo_dic= mapeo.set_index('Código')['Descripción'].to_dict()
  df_selec1[col+'_soc']= df_selec1[col].astype(str).map(mapeo_dic)

In [ ]:
cols_soc= [col for col in df_selec1.columns if col.endswith("_soc")]
print(tabulate(df_selec1[cols_soc].head(), headers='keys', tablefmt='psql'))

+----+----------------------------------------+------------------------------------------------------------+--------------------------+--------------------------------------------------------+-----------------------------------------------------------------------------------------------------+
|    | Comunidad Autónoma de residencia_soc   | Identificación del adulto seleccionado: Sexo 01 a 15_soc   | País de nacimiento_soc   | Nivel de estudios del adulto seleccionado_soc          | Variable derivada: Clase social basada en la ocupación de la persona de referencia_soc              |
|----+----------------------------------------+------------------------------------------------------------+--------------------------+--------------------------------------------------------+-----------------------------------------------------------------------------------------------------|
|  0 | País Vasco                             | Hombre                                                     | Nacido

In [ ]:
#Se copia tambien la edad

df_selec1['Edad_soc']= df_selec1['Identificación del adulto seleccionado: Edad. De 015 a 120']

In [ ]:
print(tabulate(df_selec1.head(), headers='keys', tablefmt='psql'))

+----+------------------------------------+--------------------------------------------------------+--------------------------------------------------------------+----------------------+---------------------------------------------+-----------------------------------------------------+---------------------------------------------------------------+----------------------------------------+------------------------------------+-------------------------------------------+----------------------------------------------------------------------------+----------------------------------------------------------------------------------+------------------------------------------------------------+-------------------------------------------------------------------------+------------------------------------+-------------------------------------------------------------+-------------------------------------------------+--------------------------------------------------------------------+---------------

# Manejo de NAs y duplicados

In [ ]:
# Se buscan filas duplicadas

duplicados= df_selec1.duplicated()
print(duplicados.sum())

0


No se se encuntran filas duplicadas

In [ ]:
# Se revisan los tipos de columnas
print(tabulate(df_selec1.dtypes.to_frame(), headers='keys', tablefmt='psql'))

+----------------------------------------------------------------------------------------+---------+
|                                                                                        | 0       |
|----------------------------------------------------------------------------------------+---------|
| Comunidad Autónoma de residencia                                                       | int64   |
| Identificación del adulto seleccionado: Sexo 01 a 15                                   | int64   |
| Identificación del adulto seleccionado: Edad. De 015 a 120                             | int64   |
| País de nacimiento                                                                     | int64   |
| Nivel de estudios del adulto seleccionado                                              | int64   |
| Estado de salud percibido en los últimos 12 meses                                      | int64   |
| Enfermedad o problema de salud crónicos o de larga duración                            | 

In [ ]:
nas_encontrados=df_selec1.isna().sum().sort_values(ascending=False)
print(tabulate(nas_encontrados.to_frame(), headers='keys', tablefmt='psql'))

+----------------------------------------------------------------------------------------+------+
|                                                                                        |    0 |
|----------------------------------------------------------------------------------------+------|
| Sedentarismo : tiempo que permanece sentado a lo largo de un día normal. Horas         | 1868 |
| Sedentarismo_horas_dia                                                                 | 1868 |
| IMC                                                                                    |  997 |
| IMC_cat                                                                                |  997 |
| Peso                                                                                   |  867 |
| Altura                                                                                 |  541 |
| Ha padecido alguna vez: Tensión alta_bin                                               |  308 |
| Ha padecido alguna

In [ ]:
## Si hay columnas que tengan la mitad o mas de NAs por lo que se decide eliminar basados en el porcentaje segun https://ngugijoan.medium.com/handling-missing-values-data-science-7b8e302264ee

porcentaje_limite= 0.5
columnas_a_eliminar= nas_encontrados[nas_encontrados/df_selec.shape[0] >= porcentaje_limite].index.tolist()
print(columnas_a_eliminar)

[]


In [ ]:
# No hay columnas a eliminar, por lo que se revisan las filas
na_filas= df_selec1.isna().mean(axis=1)
print(na_filas.describe())

count    21032.000000
mean         0.010569
std          0.037683
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max          0.580000
dtype: float64


In [ ]:
# Se encuentra que hay por lo menos una fila que tiene hasta el 55% de  NAs, por lo que tambien se eliminaran las filas que tengan mas del 50% de NAs

filas_a_eliminar= na_filas[na_filas >=0.5].index.tolist()
len(filas_a_eliminar)

22

In [ ]:
# Actual tamaño del conjunto de datos
df_selec1.shape

(21032, 50)

In [ ]:
# Se eliminan 15 filas, por tener mas del 50% de las respuestas nulas

df_selec1= df_selec1.drop(filas_a_eliminar)
df_selec1.shape

(21010, 50)

# Imputación de datos

Para las diferentes variables, se deciden diferentes tipos de imputacion durante el proceso de modelado predictivo. Por el tipo de variables, se determina, con la información disponible hasta el momento que:



*  Para variables de enfermedades, que son binarias, se imputara usando la moda
*  Para actividad fisica se agrega la categoria "Desconocido"
*  Para variables de cantidad y frecuencia se usar la mediana



# Descarga conjunto de datos EDA y modelado



In [ ]:
df_selec1.columns

Index(['Comunidad Autónoma de residencia',
       'Identificación del adulto seleccionado: Sexo 01 a 15',
       'Identificación del adulto seleccionado: Edad. De 015 a 120',
       'País de nacimiento', 'Nivel de estudios del adulto seleccionado',
       'Estado de salud percibido en los últimos 12 meses',
       'Enfermedad o problema de salud crónicos o de larga duración',
       'Ha padecido alguna vez: Tensión alta',
       'Ha padecido alguna vez: Diabetes',
       'Ha padecido alguna vez: Colesterol alto', 'Altura en cm (50 a 220)',
       'Peso en kg (25 a 180)',
       'Frecuencia con la que realiza alguna actividad física en su tiempo libre',
       'Sedentarismo : tiempo que permanece sentado a lo largo de un día normal. Horas',
       'Frecuencia de consumo de fruta fresca (excluyendo zumos)',
       'Frecuencia de consumo de carne (pollo, ternera, cerdo, cordero, etc.)',
       'Frecuencia de consumo de pescado',
       'Frecuencia de consumo de verduras, ensaladas y horta

In [ ]:
# Se seleccionan las variables finales

var_finals= ['Comunidad Autónoma de residencia_soc',
       'Identificación del adulto seleccionado: Sexo 01 a 15_soc',
       'País de nacimiento_soc',
       'Nivel de estudios del adulto seleccionado_soc',
       'Variable derivada: Clase social basada en la ocupación de la persona de referencia_soc',
       'Edad_soc','Frecuencia de consumo de carne (pollo, ternera, cerdo, cordero, etc.)_frec',
       'Frecuencia de consumo de productos lácteos (leche, queso, yogur)_frec',
       'Frecuencia de consumo de pescado_frec',
       'Frecuencia de consumo de fruta fresca (excluyendo zumos)_frec',
       'Frecuencia de consumo de embutidos y fiambres_frec',
       'Frecuencia de consumo de verduras, ensaladas y hortalizas_frec',
       'Frecuencia de consumo de refrescos con azúcar_frec',
       'Alcohol_cantidad', 'Salud percibida_cat',
       'Enfermedad o problema de salud crónicos o de larga duración_bin',
       'Ha padecido alguna vez: Tensión alta_bin',
       'Ha padecido alguna vez: Diabetes_bin',
       'Ha padecido alguna vez: Colesterol alto_bin', 'Comorbilidad', 'Peso',
       'Altura', 'IMC', 'Actividad_física_cat', 'Sedentarismo_horas_dia',
       'IMC_cat']

In [ ]:
df_final= df_selec1[var_finals]

In [ ]:
df_final.shape

(21010, 26)

In [ ]:
# Para mejorar la legibilidad, se cambia el nombre de las variables a variables mas cortas


df_final.rename(columns={"Frecuencia de consumo de carne (pollo, ternera, cerdo, cordero, etc.)_frec": "Carne_frec",
       "Frecuencia de consumo de productos lácteos (leche, queso, yogur)_frec": "Lacteos_frec",
        "Frecuencia de consumo de pescado_frec": "Pescado_frec",
       "Frecuencia de consumo de fruta fresca (excluyendo zumos)_frec":"Fruta_frec",
       "Frecuencia de consumo de embutidos y fiambres_frec":"Embutidos_frec",
       "Frecuencia de consumo de verduras, ensaladas y hortalizas_frec": "Verduras_frec",
       "Frecuencia de consumo de refrescos con azúcar_frec":"Refrescos_frec",
       "Alcohol_cantidad":"Alcohol_cant", "Salud percibida_cat": "Salud_Percibida",
       "Enfermedad o problema de salud crónicos o de larga duración_bin":"Cronicidad_bin",
       "Ha padecido alguna vez: Tensión alta_bin":"Hipertension_bin",
       "Ha padecido alguna vez: Diabetes_bin":"Diabetes_bin",
       "Ha padecido alguna vez: Colesterol alto_bin":"Colesterol_bin", "Comorbilidad":"Comorbilidad_num",
       "Actividad_física_cat":"Actividad_física_cat", "Sedentarismo_horas_dia": "Sedentarismo%_horas", "IMC_cat":"IMC_cat",
       "Comunidad Autónoma de residencia_soc":"Comunidad Autonoma",
       "Identificación del adulto seleccionado: Sexo 01 a 15_soc": "Sexo",
       "País de nacimiento_soc": "Pais",
       "Nivel de estudios del adulto seleccionado_soc":"Estudios", "Edad_soc":"Edad",
       "Variable derivada: Clase social basada en la ocupación de la persona de referencia_soc": "Clase"}, inplace=True)

/tmp/ipython-input-2270832569.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_final.rename(columns={"Frecuencia de consumo de carne (pollo, ternera, cerdo, cordero, etc.)_frec": "Carne_frec",


In [ ]:
print(tabulate(df_final.head(), headers='keys', tablefmt='psql'))

+----+----------------------+--------+--------------------------+--------------------------------------------------------+-----------------------------------------------------------------------------------------------------+--------+--------------+----------------+----------------+--------------+------------------+-----------------+------------------+----------------+-------------------+------------------+--------------------+----------------+------------------+--------------------+--------+----------+---------+------------------------+-----------------------+-----------+
|    | Comunidad Autonoma   | Sexo   | Pais                     | Estudios                                               | Clase                                                                                               |   Edad |   Carne_frec |   Lacteos_frec |   Pescado_frec |   Fruta_frec |   Embutidos_frec |   Verduras_frec |   Refrescos_frec |   Alcohol_cant | Salud_Percibida   |   Cronicidad_bin |   Hipertens

In [ ]:
df_final.to_csv("df_final.csv", index=False, encoding='latin1')
files.download("df_final.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>